In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import time
import logging

# Configuración de la API Key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Configuración de logging para trazabilidad
logging.basicConfig(
    filename="reparaya_logs.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Base de datos interna (simulación)
historial_clientes = {
    "juan pérez - toyota yaris 2018": "Cambio de aceite (01/02/2025), Pastillas de freno (15/03/2025)",
    "maría lópez - nissan versa 2020": "Revisión general (20/02/2025)"
}

tarifas = {
    "cambio de pastillas de freno": "45.000 CLP",
    "cambio de aceite": "25.000 CLP",
    "revisión general": "30.000 CLP"
}

# Métricas de observabilidad
metricas = {
    "consultas_totales": 0,
    "latencia_promedio": [],
    "errores": 0
}

# Función para responder consultas
def asistente_reparaya(pregunta):
    inicio = time.time()
    pregunta_lower = pregunta.lower()
    contexto = ""

    # Buscar en el historial de clientes
    for cliente, datos in historial_clientes.items():
        if cliente in pregunta_lower:
            contexto += f"Historial de {cliente}: {datos}\n"

    # Buscar tarifas de servicios
    for servicio, precio in tarifas.items():
        if servicio in pregunta_lower:
            contexto += f"Tarifa {servicio}: {precio}\n"

    if not contexto:
        contexto = "No encontré información interna relevante."

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",   # Usa "gpt-4o" si tienes acceso completo
            messages=[
                {"role": "system", "content": "Eres un asistente del taller mecánico ReparaYA. Usa solo los datos del contexto."},
                {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta del cliente:\n{pregunta}"}
            ],
            max_tokens=200
        )
        texto = response.choices[0].message.content.strip()
    except Exception as e:
        logging.error(f"Error en la API: {e}")
        metricas["errores"] += 1
        return "Lo siento, hubo un error al procesar tu solicitud."

    # Actualizar métricas
    fin = time.time()
    latencia = fin - inicio
    metricas["consultas_totales"] += 1
    metricas["latencia_promedio"].append(latencia)

    # Log de trazabilidad
    logging.info(f"Consulta: {pregunta} | Latencia: {latencia:.2f}s | Contexto usado: {contexto}")

    return texto

# Función para mostrar métricas y recomendaciones
def reporte_observabilidad():
    if metricas["consultas_totales"] == 0:
        return "No hay métricas disponibles aún."

    latencia_media = sum(metricas["latencia_promedio"]) / len(metricas["latencia_promedio"])
    reporte = f"""
📊 Reporte de Observabilidad:
- Consultas totales: {metricas['consultas_totales']}
- Latencia promedio: {latencia_media:.2f} segundos
- Errores registrados: {metricas['errores']}

🔎 Recomendaciones:
- Optimizar caché de respuestas frecuentes para reducir latencia.
- Revisar logs en 'reparaya_logs.log' para detectar cuellos de botella.
- Escalar infraestructura si la latencia supera consistentemente los 2s.
"""
    return reporte

# Ejemplo de uso
if __name__ == "__main__":
    preguntas = [
        "¿Cuál es el historial de reparaciones de Juan Pérez - Toyota Yaris 2018?",
        "¿Cuánto cuesta un cambio de pastillas de freno?",
        "¿Puedo agendar una cita para el lunes a las 10:00 hrs?"
    ]

    print("🤖 Asistente ReparaYA\n")
    for p in preguntas:
        print(f"❓ {p}")
        print("✅", asistente_reparaya(p))
        print("-" * 50)

    # Mostrar métricas finales
    print(reporte_observabilidad())


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable